# BIONIC DAUGHTER HACKER — GRPO Training on Free T4 GPU

**Optimized for Google Colab VS Code Extension (free GPU, no credit card)**

- **Base model:** Qwen3-4B-Thinking-2507 (Apache 2.0, no token needed)
- **Dataset:** 4,750 examples across 33 domains
- **Rewards:** 8 hacker-style rewards (format, accuracy, reasoning, creativity, attack_chain, bypass, tools, self-correction)
- **Hardware:** Free T4 GPU (16GB VRAM) via Colab
- **Time:** ~2-3 hours for 400 GRPO steps

In [ ]:
# CELL 1: Install dependencies (optimized for T4)
!pip install -q unsloth trl peft datasets accelerate bitsandbytes huggingface_hub
!pip install -q jsonlines

In [ ]:
# CELL 2: Clone training data and reward engine from your repo
!git clone https://github.com/Mmarcos815/bionic-daughter-training.git /content/repo
!cp /content/repo/grpo_train_final.jsonl /content/data.jsonl
!cp /content/repo/grpo_reward_engine.py /content/reward_engine.py
!cp /content/repo/grpo_training_config.yaml /content/config.yaml
print("Data and config loaded!")

In [ ]:
# CELL 3: Verify dataset
import json

data = []
with open('/content/data.jsonl') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"Loaded {len(data)} training examples")

# Show domain distribution
from collections import Counter
domains = Counter(d.get('metadata', {}).get('domain', 'unknown') for d in data)
print(f"\nDomain coverage: {len(domains)} domains")
for domain, count in domains.most_common(10):
    print(f"  {domain}: {count}")

In [ ]:
# CELL 4: Download base model (no token needed — Apache 2.0)
import os
os.system("git lfs install")
!git clone --depth 1 https://huggingface.co/Qwen/Qwen3-4B-Thinking-2507 /content/base_model
print("Model downloaded!")

In [ ]:
# CELL 5: Load reward engine
import sys
sys.path.insert(0, '/content')
from grpo_reward_engine import (
    reward_format,
    reward_accuracy,
    reward_reasoning_depth,
    reward_creativity,
    reward_attack_chain,
    reward_bypass_creativity,
    reward_tool_use_quality,
    reward_self_correction,
    DEFAULT_REWARD_WEIGHTS,
)

# Wrapper functions for TRL
def reward_format_wrapper(completions, **kwargs):
    return [reward_format(c, DEFAULT_REWARD_WEIGHTS) for c in completions]

def reward_accuracy_wrapper(completions, **kwargs):
    prompts = kwargs.get('prompts', [''] * len(completions))
    return [reward_accuracy(c, p, DEFAULT_REWARD_WEIGHTS) for c, p in zip(completions, prompts)]

def reward_reasoning_wrapper(completions, **kwargs):
    return [reward_reasoning_depth(c, DEFAULT_REWARD_WEIGHTS) for c in completions]

def reward_creativity_wrapper(completions, **kwargs):
    return [reward_creativity(c, '', DEFAULT_REWARD_WEIGHTS) for c in completions]

def reward_attack_chain_wrapper(completions, **kwargs):
    return [reward_attack_chain(c, '', DEFAULT_REWARD_WEIGHTS) for c in completions]

def reward_bypass_wrapper(completions, **kwargs):
    return [reward_bypass_creativity(c, '', DEFAULT_REWARD_WEIGHTS) for c in completions]

def reward_tools_wrapper(completions, **kwargs):
    prompts = kwargs.get('prompts', [''] * len(completions))
    return [reward_tool_use_quality(c, p, DEFAULT_REWARD_WEIGHTS) for c, p in zip(completions, prompts)]

def reward_self_corr_wrapper(completions, **kwargs):
    return [reward_self_correction(c, DEFAULT_REWARD_WEIGHTS) for c in completions]

print("Reward engine loaded with 8 hacker rewards!")

In [ ]:
# CELL 6: Load model with Unsloth (4-bit quantized for T4)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/base_model",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("Model loaded with LoRA (4-bit, T4-optimized)!")

In [ ]:
# CELL 7: Load dataset
from datasets import Dataset

data = []
with open('/content/data.jsonl') as f:
    for line in f:
        data.append(json.loads(line.strip()))

dataset = Dataset.from_list(data)
print(f"Dataset: {len(dataset)} examples")

In [ ]:
# CELL 8: Configure GRPO training (T4-optimized)
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="/content/output",
    num_train_epochs=1,
    max_steps=400,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=50,
    save_total_limit=3,
    bf16=True,
    beta=0.01,
    max_prompt_length=1024,
    max_completion_length=1024,
    temperature=0.9,
    top_p=0.95,
    top_k=50,
    num_generations=6,
    report_to="none",
)

In [ ]:
# CELL 9: Initialize trainer with 8 hacker rewards
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_format_wrapper,
        reward_accuracy_wrapper,
        reward_reasoning_wrapper,
        reward_creativity_wrapper,
        reward_attack_chain_wrapper,
        reward_bypass_wrapper,
        reward_tools_wrapper,
        reward_self_corr_wrapper,
    ],
    args=training_args,
    train_dataset=dataset,
)

print("Trainer ready with 8 hacker rewards!")

In [ ]:
# CELL 10: TRAIN!
trainer.train()
print("\nTraining complete!")

In [ ]:
# CELL 11: Save trained model
model.save_pretrained("/content/bionic-daughter-hacker")
tokenizer.save_pretrained("/content/bionic-daughter-hacker")
print("Model saved to /content/bionic-daughter-hacker")

# Also save to Google Drive if mounted
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r /content/bionic-daughter-hacker /content/drive/MyDrive/
    print("Model saved to Google Drive!")
except:
    print("Drive not mounted — model saved to /content only")

In [ ]:
# CELL 12: Test the trained model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/bionic-daughter-hacker",
    max_seq_length=2048,
    load_in_4bit=True,
)

test_prompts = [
    "Analyze this SQL injection vulnerability in a login form",
    "How would you chain a CSRF with a stored XSS for account takeover?",
    "What is the MITRE ATT&CK technique for LLMNR poisoning?",
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Response: {response[:300]}")